In [ ]:
!pip -q install dspy-ai requests

In [2]:
import os
import requests
import dspy
from dataclasses import dataclass
from typing import Any, Dict, List, Optional
import re
from google.colab import userdata


# ---- USER FILLS THESE IN ----
AZURE_API_KEY = userdata.get('Azure')          # <-- put your key in Colab secrets or env var
AZURE_API_BASE = "https://o3miniapi.cognitiveservices.azure.com/"  # your endpoint
AZURE_API_VERSION = "2024-12-01-preview"
AZURE_DEPLOYMENT = "gpt-5-mini"       # <-- your chat-capable deployment name (e.g., "gpt-4o-mini")

# Recommended: use env var rather than hardcoding.
if AZURE_API_KEY:
    os.environ["AZURE_API_KEY"] = AZURE_API_KEY

# DSPy LM via LiteLLM Azure provider string.
# Note: provider strings can vary across LiteLLM versions; "azure/<deployment>" is the common pattern.
lm = dspy.LM(
    f"azure/{AZURE_DEPLOYMENT}",
    api_key=os.getenv("AZURE_API_KEY", ""),
    api_base=AZURE_API_BASE,
    api_version=AZURE_API_VERSION,
    model_type="chat",
    temperature=1.0,
    max_tokens=32000,
)

dspy.configure(lm=lm)

In [3]:
import logging

# More verbose internal logs (optional but useful)
logging.getLogger("dspy").setLevel(logging.DEBUG)  # DSPy FAQ mentions this approach. :contentReference[oaicite:1]{index=1}

# Track token usage (DSPy 2.6.16+)
dspy.configure(track_usage=True)  # usage can be read from Prediction.get_lm_usage(). :contentReference[oaicite:2]{index=2}

In [4]:
# -------------------------
# 2) data.gouv.fr REST client (tool)
# -------------------------
DATASETS_URL = "https://www.data.gouv.fr/api/1/datasets/"

def search_datasets(
    q: str,
    page_size: int = 10,
    page: int = 1,
    sort: str = "-last_update",
    tag: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Calls: GET /api/1/datasets/?q=...&page_size=...&page=...&sort=...
    """
    params: Dict[str, Any] = {
        "q": q,
        "page_size": page_size,
        "page": page,
        "sort": sort,
    }
    if tag:
        params["tag"] = tag

    r = requests.get(DATASETS_URL, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def _trim(s: Optional[str], n: int = 280) -> Optional[str]:
    if not s:
        return s
    s = re.sub(r"\s+", " ", s).strip()
    return s if len(s) <= n else s[: n - 1] + "…"

def _formats_from_resources(ds: Dict[str, Any], max_formats: int = 6) -> List[str]:
    fmts = []
    for r in (ds.get("resources") or []):
        fmt = (r.get("format") or "").strip()
        if fmt:
            fmts.append(fmt.upper())
    # unique preserve order
    seen = set()
    out = []
    for f in fmts:
        if f not in seen:
            seen.add(f)
            out.append(f)
    return out[:max_formats]


In [5]:
# -------------------------
# 3) DSPy signature: plan the API search
# -------------------------
class PlanDatasetSearch(dspy.Signature):
    """
    Turn a user's question into a compact data.gouv.fr dataset search plan.
    Keep q SHORT. Use sort for "recent" instead of stuffing years into q.
    """
    question: str = dspy.InputField()

    q: str = dspy.OutputField(desc="2–6 keywords in French or English. No years. No long phrases.")
    sort: str = dspy.OutputField(desc="One of: -last_update, last_update, -created, created, -views, views")
    page_size: int = dspy.OutputField(desc="Integer 1–50")
    tag: Optional[str] = dspy.OutputField(desc="Optional tag to filter (or null/None)")


In [6]:
# -------------------------
# 4) Utility: fallback query simplification
# -------------------------
def simplify_query(q: str) -> str:
    """
    If the LM makes q too verbose, strip filler and years.
    """
    q = (q or "").strip().lower()
    q = re.sub(r"\b(19|20)\d{2}\b", " ", q)  # remove years like 2020, 2024
    filler = {
        "donnees", "données", "data", "dataset", "datasets",
        "france", "français", "francaise", "française",
        "recent", "récents", "récent", "recentes", "récentes",
        "preferably", "prefer", "preferably",
        "in", "of", "the", "and", "about",
        "sur", "de", "des", "du", "la", "le", "les", "un", "une"
    }
    tokens = [t for t in re.split(r"\W+", q) if t and t not in filler]
    # keep a few tokens only
    return " ".join(tokens[:5]) if tokens else q.strip()


In [7]:
# -------------------------
# 5) The DSPy "agent" module
# -------------------------
class DataGouvSearchAgent(dspy.Module):
    def __init__(self, default_page_size: int = 10):
        super().__init__()
        self.plan = dspy.Predict(PlanDatasetSearch)
        self.default_page_size = default_page_size

    def forward(self, question: str, page: int = 1) -> dspy.Prediction:
        # 1) LM plans the API call
        plan = self.plan(question=question)

        # 2) sanitize plan fields
        q = (plan.q or "").strip()
        sort = (plan.sort or "-last_update").strip()

        # constrain sort to allowed values
        allowed_sorts = {"-last_update", "last_update", "-created", "created", "-views", "views"}
        if sort not in allowed_sorts:
            sort = "-last_update"

        # constrain page_size
        try:
            page_size = int(plan.page_size)
        except Exception:
            page_size = self.default_page_size
        page_size = max(1, min(page_size, 50))

        tag = plan.tag if plan.tag else None

        # 3) Call API (first attempt)
        results = search_datasets(q=q, page_size=page_size, page=page, sort=sort, tag=tag)

        # 4) Fallback: if zero results, try simplified q (and drop tag)
        total = results.get("total") or 0
        fallback_used = False
        if total == 0:
            q2 = simplify_query(q)
            if q2 and q2 != q:
                fallback_used = True
                results = search_datasets(q=q2, page_size=page_size, page=page, sort=sort, tag=None)
                q = q2
                tag = None
                total = results.get("total") or 0

        # 5) Build compact dataset cards
        datasets: List[Dict[str, Any]] = []
        for ds in results.get("data", []):
            org = ds.get("organization") or {}
            datasets.append({
                "id": ds.get("id"),
                "title": ds.get("title"),
                "organization": org.get("name"),
                "last_update": ds.get("last_update"),
                "created_at": ds.get("created_at"),
                "description": _trim(ds.get("description"), 320),
                "formats": _formats_from_resources(ds),
                "uri": ds.get("uri"),  # usually the dataset page URL
            })

        # 6) Return dspy.Prediction (fixes usage tracking warning)
        return dspy.Prediction(
            question=question,
            planned_query={
                "q": q,
                "page_size": page_size,
                "page": page,
                "sort": sort,
                "tag": tag,
                "fallback_used": fallback_used,
            },
            total=total,
            datasets=datasets,
        )


In [ ]:
# -------------------------
# 6) Try it
# -------------------------
agent = DataGouvSearchAgent(default_page_size=15)

out = agent(question="I need datasets about traffic accidents in France, preferably recent.")
out


Prediction(
    question='I need datasets about traffic accidents in France, preferably recent.',
    planned_query={'q': 'accidents routiers', 'page_size': 20, 'page': 1, 'sort': '-last_update', 'tag': None, 'fallback_used': True},
    total=1,
    datasets=[{'id': '67f073c59532cfae465b7d1c', 'title': 'Localisation des accidents routiers sur la Métropole TPM ', 'organization': 'Métropole Toulon Provence Méditerranée', 'last_update': '2025-04-04T00:00:00+00:00', 'created_at': '2025-03-27T00:00:00+00:00', 'description': "Ce jeu de données est une extraction des caractéristiques essentielles des accidents routiers. Il permet d'avoir une vision simplifiée des accidents **Lien vers la[dataviz ](https://portailsig.metropoletpm.fr/portal/apps/instant/portfolio/index.html?appid=9adb2fb78e124e09b2b104fdc47a6222)** Si plusieurs usagers sont i…", 'formats': ['WEB PAGE', 'ARCGIS GEOSERVICES REST API', 'CSV', 'ZIP', 'GEOJSON', 'KML'], 'uri': 'https://www.data.gouv.fr/api/1/datasets/localisation-de

In [ ]:
# Pretty-print the first few results nicely (optional)
def show_results(pred: dspy.Prediction, top_k: int = 5):
    print("Question:", pred.question)
    print("Planned query:", pred.planned_query)
    print("Total:", pred.total)
    print("\nTop results:")
    for i, ds in enumerate(pred.datasets[:top_k], 1):
        print(f"\n#{i} {ds.get('title')}")
        print("   org:", ds.get("organization"))
        print("   last_update:", ds.get("last_update"))
        print("   formats:", ds.get("formats"))
        print("   uri:", ds.get("uri"))
        print("   desc:", ds.get("description"))

show_results(out, top_k=5)


Question: I need datasets about traffic accidents in France, preferably recent.
Planned query: {'q': 'accidents routiers', 'page_size': 20, 'page': 1, 'sort': '-last_update', 'tag': None, 'fallback_used': True}
Total: 1

Top results:

#1 Localisation des accidents routiers sur la Métropole TPM 
   org: Métropole Toulon Provence Méditerranée
   last_update: 2025-04-04T00:00:00+00:00
   formats: ['WEB PAGE', 'ARCGIS GEOSERVICES REST API', 'CSV', 'ZIP', 'GEOJSON', 'KML']
   uri: https://www.data.gouv.fr/api/1/datasets/localisation-des-accidents-routiers-sur-la-metropole-tpm/
   desc: Ce jeu de données est une extraction des caractéristiques essentielles des accidents routiers. Il permet d'avoir une vision simplifiée des accidents **Lien vers la[dataviz ](https://portailsig.metropoletpm.fr/portal/apps/instant/portfolio/index.html?appid=9adb2fb78e124e09b2b104fdc47a6222)** Si plusieurs usagers sont i…


In [ ]:
print("LM history length:", len(lm.history), "   --> (Note: this is probably number of times that the LLM was called IN TOTAL - not just in the last run)")
print("Last call keys:", lm.history[-1].keys())
print("Last call usage:", lm.history[-1].get("usage"))
print("Last call cost:", lm.history[-1].get("cost"))

LM history length: 1    --> (Note: this is probably number of times that the LLM was called IN TOTAL - not just in the last run)
Last call keys: dict_keys(['prompt', 'messages', 'kwargs', 'response', 'outputs', 'usage', 'cost', 'timestamp', 'uuid', 'model', 'response_model', 'model_type'])
Last call usage: {}
Last call cost: 0.00147125


In [ ]:
# Direct API sanity checks (no DSPy)
print(search_datasets(q="accidents route", page_size=5, sort="-last_update").get("total"))
print(search_datasets(q="sécurité routière", page_size=5, sort="-last_update").get("total"))
print(search_datasets(q="baac", page_size=5, sort="-last_update").get("total"))


5
4
0


So this tells us one thing: That the API search query is TERRIBLE, and that we are better off using the Vector DB.